# RNN vs Transformer Assignment Solution

Rename this notebook to `YOUR_BITS_ID_rnn_assignment.ipynb` before submission. Fill in the student details below so the notebook matches your final submission.

## Student Information
- BITS ID: REPLACE_ME
- Name: REPLACE_ME
- Email: REPLACE_ME
- Date: REPLACE_ME

## Approach
- Dataset: Daily minimum temperatures weather series
- Framework: PyTorch
- RNN model: 2-layer stacked LSTM
- Transformer model: TransformerEncoder with sinusoidal positional encoding
- Split: 85/15 temporal split with no shuffling

In [ ]:
# If you are running in a fresh Colab or VM, install the required packages first.
# !pip install torch torchvision pandas numpy scikit-learn seaborn matplotlib tqdm

In [1]:
from __future__ import annotations

import copy
import json
import math
import platform
import random
import subprocess
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import MinMaxScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm

sns.set_theme(style='whitegrid')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
)
print(f'Using device: {device}')

student_info = {
    'BITS ID': 'REPLACE_ME',
    'Name': 'REPLACE_ME',
    'Email': 'REPLACE_ME',
    'Date': 'REPLACE_ME',
}
student_info

Using device: mps


{'BITS ID': 'REPLACE_ME',
 'Name': 'REPLACE_ME',
 'Email': 'REPLACE_ME',
 'Date': 'REPLACE_ME'}

In [ ]:
DATA_ROOT = Path('assignment_data')
DATA_ROOT.mkdir(exist_ok=True)

DATA_URL = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/daily-min-temperatures.csv'
DATA_PATH = DATA_ROOT / 'daily-min-temperatures.csv'

def download_file(url: str, destination: Path) -> Path:
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists():
        return destination
    print(f'Downloading {url} -> {destination}')
    subprocess.run(['curl', '-L', '-sS', url, '-o', str(destination)], check=True)
    return destination

download_file(DATA_URL, DATA_PATH)
df = pd.read_csv(DATA_PATH, parse_dates=['Date'])
df.columns = ['date', 'temperature']
df = df.sort_values('date').reset_index(drop=True)

dataset_name = 'Daily Minimum Temperatures'
dataset_source = DATA_URL
n_samples = int(len(df))
n_features = 1
sequence_length = 30
prediction_horizon = 1
problem_type = 'time_series_forecasting'
primary_metric = 'RMSE'
metric_justification = (
    'RMSE is the primary metric because the forecast target is continuous temperature in the original unit scale. '
    'It penalizes larger forecast misses more strongly than MAE, which is useful for time-series prediction.'
)

print('DATASET INFORMATION')
print(f'Dataset: {dataset_name}')
print(f'Source: {dataset_source}')
print(f'Total Samples: {n_samples}')
print(f'Number of Features: {n_features}')
print(f'Sequence Length: {sequence_length}')
print(f'Prediction Horizon: {prediction_horizon}')
print(f'Primary Metric: {primary_metric}')
print(f'Metric Justification: {metric_justification}')
df.head()

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0

DATASET INFORMATION
Dataset: Daily Minimum Temperatures
Source: https://raw.githubusercontent.com/jbrownlee/Datasets/master/daily-min-temperatures.csv
Total Samples: 3650
Number of Features: 1
Sequence Length: 30
Prediction Horizon: 1
Primary Metric: RMSE
Metric Justification: RMSE is the primary metric because the forecast target is continuous temperature in the original unit scale. It penalizes larger forecast misses more strongly than MAE, which is useful for time-series prediction.


100 67921  100 67921    0     0   191k      0 --:--:-- --:--:-- --:--:--  191k


,date,temperature
0,1981-01-01,20.7
1,1981-01-02,17.9
2,1981-01-03,18.8
3,1981-01-04,14.6
4,1981-01-05,15.8


In [ ]:
temperature_series = df['temperature'].astype('float32')
rolling_window = 30
rolling_mean = temperature_series.rolling(window=rolling_window).mean()
rolling_std = temperature_series.rolling(window=rolling_window).std()
first_difference = temperature_series.diff()

fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=False)
axes[0].plot(df['date'], temperature_series, color='steelblue', linewidth=1)
axes[0].set_title('Daily Minimum Temperature Time Series')
axes[0].set_ylabel('Temperature')

axes[1].plot(df['date'], temperature_series, label='Original', alpha=0.7)
axes[1].plot(df['date'], rolling_mean, label='30-day Rolling Mean', color='crimson')
axes[1].plot(df['date'], rolling_std, label='30-day Rolling Std', color='darkgreen')
axes[1].set_title('Rolling Statistics for Stationarity Analysis')
axes[1].legend()

axes[2].plot(df['date'], first_difference, color='purple', linewidth=1)
axes[2].set_title('First Difference of the Series')
axes[2].set_xlabel('Date')
axes[2].set_ylabel('Differenced Temperature')

plt.tight_layout()
plt.show()

stationarity_summary = pd.DataFrame({
    'Statistic': ['Original Mean', 'Original Std', 'First Difference Mean', 'First Difference Std'],
    'Value': [
        float(temperature_series.mean()),
        float(temperature_series.std()),
        float(first_difference.dropna().mean()),
        float(first_difference.dropna().std()),
    ],
})
stationarity_summary

In [ ]:
def preprocess_timeseries(data: np.ndarray):
    values = np.asarray(data, dtype=np.float32).reshape(-1, 1)
    values = np.nan_to_num(values, nan=float(np.nanmean(values)))
    scaler = MinMaxScaler()
    scaled = scaler.fit_transform(values)
    return scaled, scaler

def create_sequences(data: np.ndarray, seq_length: int, pred_horizon: int):
    sequences = []
    targets = []
    target_indices = []

    for end_idx in range(seq_length, len(data) - pred_horizon + 1):
        start_idx = end_idx - seq_length
        target_end = end_idx + pred_horizon
        sequences.append(data[start_idx:end_idx])
        targets.append(data[end_idx:target_end, 0])
        target_indices.append(target_end - 1)

    return (
        np.asarray(sequences, dtype=np.float32),
        np.asarray(targets, dtype=np.float32),
        np.asarray(target_indices, dtype=np.int32),
    )

raw_values = df[['temperature']].to_numpy(dtype=np.float32)
split_index = int(len(raw_values) * 0.85)

scaler = MinMaxScaler()
scaler.fit(raw_values[:split_index])
scaled_values = scaler.transform(raw_values)

X_all, y_all, target_indices = create_sequences(scaled_values, sequence_length, prediction_horizon)
train_mask = target_indices < split_index
test_mask = target_indices >= split_index

X_train_full = X_all[train_mask]
y_train_full = y_all[train_mask]
X_test = X_all[test_mask]
y_test = y_all[test_mask]
test_target_indices = target_indices[test_mask]

val_size = max(1, int(len(X_train_full) * 0.10))
X_train = X_train_full[:-val_size]
y_train = y_train_full[:-val_size]
X_val = X_train_full[-val_size:]
y_val = y_train_full[-val_size:]

train_test_ratio = '85/15'
train_samples = int(len(X_train_full))
test_samples = int(len(X_test))

print(f'\nTrain/Test Split: {train_test_ratio}')
print(f'Training Samples: {train_samples}')
print(f'Validation Samples: {len(X_val)}')
print(f'Test Samples: {test_samples}')
print('IMPORTANT: Temporal split used (NO shuffling)')

boundary_date = df.iloc[split_index]['date']
plt.figure(figsize=(14, 5))
plt.plot(df['date'], df['temperature'], label='Temperature', linewidth=1.1)
plt.axvline(boundary_date, color='crimson', linestyle='--', label='Train/Test Boundary')
plt.title('Temporal Train/Test Split')
plt.xlabel('Date')
plt.ylabel('Temperature')
plt.legend()
plt.show()

In [ ]:
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=64, shuffle=False)
val_loader = DataLoader(TensorDataset(X_val_tensor, y_val_tensor), batch_size=64, shuffle=False)
test_loader = DataLoader(TensorDataset(X_test_tensor, y_test_tensor), batch_size=64, shuffle=False)

def calculate_mape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float32)
    y_pred = np.asarray(y_pred, dtype=np.float32)
    denominator = np.maximum(np.abs(y_true), 1e-6)
    return float(np.mean(np.abs((y_true - y_pred) / denominator)) * 100.0)

def inverse_transform_targets(array: np.ndarray, scaler: MinMaxScaler) -> np.ndarray:
    array_2d = np.asarray(array, dtype=np.float32).reshape(-1, 1)
    return scaler.inverse_transform(array_2d).reshape(-1)

def count_parameters(model: nn.Module) -> int:
    return sum(parameter.numel() for parameter in model.parameters())

def run_epoch(model, loader, criterion, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss = 0.0

    for inputs, targets in loader:
        inputs = inputs.to(device)
        targets = targets.to(device)

        with torch.set_grad_enabled(training):
            predictions = model(inputs)
            loss = criterion(predictions, targets)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * inputs.size(0)

    return total_loss / len(loader.dataset)

def train_model(model, train_loader, val_loader, criterion, optimizer, epochs):
    history = {'train_loss': [], 'val_loss': []}
    best_state = copy.deepcopy(model.state_dict())
    best_val_loss = float('inf')

    for epoch in range(1, epochs + 1):
        train_loss = run_epoch(model, train_loader, criterion, optimizer)
        val_loss = run_epoch(model, val_loader, criterion)
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())

        print(f'Epoch {epoch:02d}/{epochs} | train_loss={train_loss:.5f} | val_loss={val_loss:.5f}')

    model.load_state_dict(best_state)
    return history

def predict_model(model, loader):
    model.eval()
    outputs = []
    targets = []

    with torch.no_grad():
        for inputs, batch_targets in loader:
            predictions = model(inputs.to(device)).cpu().numpy()
            outputs.append(predictions)
            targets.append(batch_targets.numpy())

    return np.vstack(outputs), np.vstack(targets)

def evaluate_regression_model(model, loader, scaler):
    scaled_predictions, scaled_targets = predict_model(model, loader)
    y_pred = inverse_transform_targets(scaled_predictions, scaler)
    y_true = inverse_transform_targets(scaled_targets, scaler)
    metrics = {
        'mae': float(mean_absolute_error(y_true, y_pred)),
        'rmse': float(np.sqrt(mean_squared_error(y_true, y_pred))),
        'mape': float(calculate_mape(y_true, y_pred)),
        'r2_score': float(r2_score(y_true, y_pred)),
        'y_true': y_true,
        'y_pred': y_pred,
        'residuals': y_true - y_pred,
    }
    return metrics

def plot_history(history, title):
    epochs = range(1, len(history['train_loss']) + 1)
    plt.figure(figsize=(8, 4))
    plt.plot(epochs, history['train_loss'], marker='o', label='Train Loss')
    plt.plot(epochs, history['val_loss'], marker='o', label='Validation Loss')
    plt.title(f'{title} Training Curve')
    plt.xlabel('Epoch')
    plt.ylabel('MSE Loss')
    plt.legend()
    plt.show()

def plot_predictions(y_true, y_pred, title, max_points=220):
    limit = min(max_points, len(y_true))
    plt.figure(figsize=(14, 5))
    plt.plot(y_true[:limit], label='Actual', linewidth=2)
    plt.plot(y_pred[:limit], label='Predicted', linewidth=2)
    plt.title(title)
    plt.xlabel('Test Time Step')
    plt.ylabel('Temperature')
    plt.legend()
    plt.show()

def plot_residuals(residuals, title):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(residuals, color='slateblue')
    axes[0].axhline(0.0, color='black', linestyle='--')
    axes[0].set_title(f'{title} Residual Series')

    sns.histplot(residuals, bins=30, kde=True, ax=axes[1], color='teal')
    axes[1].set_title(f'{title} Residual Distribution')
    plt.tight_layout()
    plt.show()

In [ ]:
class StackedRNNRegressor(nn.Module):
    def __init__(self, model_type: str, input_size: int, hidden_units: int, n_layers: int, output_size: int, dropout: float = 0.2):
        super().__init__()
        self.model_type = model_type
        rnn_cls = nn.LSTM if model_type.upper() == 'LSTM' else nn.GRU
        self.recurrent = rnn_cls(
            input_size=input_size,
            hidden_size=hidden_units,
            num_layers=n_layers,
            batch_first=True,
            dropout=dropout if n_layers > 1 else 0.0,
        )
        self.head = nn.Sequential(
            nn.Linear(hidden_units, hidden_units // 2),
            nn.ReLU(),
            nn.Linear(hidden_units // 2, output_size),
        )

    def forward(self, inputs):
        outputs, _ = self.recurrent(inputs)
        final_state = outputs[:, -1, :]
        return self.head(final_state)

def build_rnn_model(model_type, input_shape, hidden_units, n_layers, output_size):
    _, n_features_local = input_shape
    return StackedRNNRegressor(model_type, n_features_local, hidden_units, n_layers, output_size).to(device)

class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 5000):
        super().__init__()
        positions = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model))
        encoding = torch.zeros(max_len, d_model, dtype=torch.float32)
        encoding[:, 0::2] = torch.sin(positions * div_term)
        encoding[:, 1::2] = torch.cos(positions * div_term)
        self.register_buffer('encoding', encoding.unsqueeze(0), persistent=False)

    def forward(self, inputs):
        return inputs + self.encoding[:, : inputs.size(1), :]

def positional_encoding(seq_length, d_model):
    positions = np.arange(seq_length)[:, np.newaxis]
    div_term = np.exp(np.arange(0, d_model, 2) * (-np.log(10000.0) / d_model))
    encoding = np.zeros((seq_length, d_model), dtype=np.float32)
    encoding[:, 0::2] = np.sin(positions * div_term)
    encoding[:, 1::2] = np.cos(positions * div_term)
    return encoding

class TransformerRegressor(nn.Module):
    def __init__(self, n_features: int, d_model: int, n_heads: int, n_layers: int, d_ff: int, output_size: int, dropout: float = 0.1):
        super().__init__()
        self.input_projection = nn.Linear(n_features, d_model)
        self.positional_encoder = PositionalEncoding(d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_ff,
            dropout=dropout,
            batch_first=True,
            activation='gelu',
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.head = nn.Sequential(
            nn.Linear(d_model, d_ff // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff // 2, output_size),
        )

    def forward(self, inputs):
        projected = self.input_projection(inputs)
        encoded = self.positional_encoder(projected)
        transformed = self.encoder(encoded)
        pooled = transformed[:, -1, :]
        return self.head(pooled)

example_positional_encoding = positional_encoding(sequence_length, 8)
plt.figure(figsize=(8, 3))
sns.heatmap(example_positional_encoding, cmap='viridis')
plt.title('Example Sinusoidal Positional Encoding (30 x 8)')
plt.xlabel('Embedding Dimension')
plt.ylabel('Sequence Position')
plt.show()

In [ ]:
print('\nRNN MODEL TRAINING')

rnn_model_type = 'LSTM'
rnn_hidden_units = 64
rnn_n_layers = 2
rnn_model = build_rnn_model(rnn_model_type, (sequence_length, n_features), rnn_hidden_units, rnn_n_layers, prediction_horizon)
rnn_total_parameters = count_parameters(rnn_model)
print(rnn_model)
print(f'RNN parameters: {rnn_total_parameters:,}')

rnn_learning_rate = 0.001
rnn_epochs = 12
rnn_batch_size = 64
rnn_optimizer_name = 'Adam'
rnn_loss_name = 'MSE'

rnn_criterion = nn.MSELoss()
rnn_optimizer = torch.optim.Adam(rnn_model.parameters(), lr=rnn_learning_rate)

rnn_start_time = time.time()
rnn_history = train_model(rnn_model, train_loader, val_loader, rnn_criterion, rnn_optimizer, rnn_epochs)
rnn_training_time = time.time() - rnn_start_time

rnn_initial_loss = float(rnn_history['train_loss'][0])
rnn_final_loss = float(rnn_history['train_loss'][-1])

print(f'Training completed in {rnn_training_time:.2f} seconds')
print(f'Initial Loss: {rnn_initial_loss:.4f}')
print(f'Final Loss: {rnn_final_loss:.4f}')

rnn_metrics = evaluate_regression_model(rnn_model, test_loader, scaler)
rnn_mae = float(rnn_metrics['mae'])
rnn_rmse = float(rnn_metrics['rmse'])
rnn_mape = float(rnn_metrics['mape'])
rnn_r2 = float(rnn_metrics['r2_score'])

print('\nRNN Model Performance:')
print(f'MAE:   {rnn_mae:.4f}')
print(f'RMSE:  {rnn_rmse:.4f}')
print(f'MAPE:  {rnn_mape:.4f}%')
print(f'R² Score: {rnn_r2:.4f}')

plot_history(rnn_history, 'RNN')
plot_predictions(rnn_metrics['y_true'], rnn_metrics['y_pred'], 'RNN: Actual vs Predicted Temperature')
plot_residuals(rnn_metrics['residuals'], 'RNN')

In [ ]:
print('\nTRANSFORMER MODEL TRAINING')

transformer_n_layers = 2
transformer_n_heads = 4
transformer_d_model = 64
transformer_d_ff = 128
transformer_model = TransformerRegressor(
    n_features=n_features,
    d_model=transformer_d_model,
    n_heads=transformer_n_heads,
    n_layers=transformer_n_layers,
    d_ff=transformer_d_ff,
    output_size=prediction_horizon,
).to(device)
transformer_total_parameters = count_parameters(transformer_model)
print(transformer_model)
print(f'Transformer parameters: {transformer_total_parameters:,}')

transformer_learning_rate = 0.001
transformer_epochs = 12
transformer_batch_size = 64
transformer_optimizer_name = 'Adam'
transformer_loss_name = 'MSE'

transformer_criterion = nn.MSELoss()
transformer_optimizer = torch.optim.Adam(transformer_model.parameters(), lr=transformer_learning_rate)

transformer_start_time = time.time()
transformer_history = train_model(transformer_model, train_loader, val_loader, transformer_criterion, transformer_optimizer, transformer_epochs)
transformer_training_time = time.time() - transformer_start_time

transformer_initial_loss = float(transformer_history['train_loss'][0])
transformer_final_loss = float(transformer_history['train_loss'][-1])

print(f'Training completed in {transformer_training_time:.2f} seconds')
print(f'Initial Loss: {transformer_initial_loss:.4f}')
print(f'Final Loss: {transformer_final_loss:.4f}')

transformer_metrics = evaluate_regression_model(transformer_model, test_loader, scaler)
transformer_mae = float(transformer_metrics['mae'])
transformer_rmse = float(transformer_metrics['rmse'])
transformer_mape = float(transformer_metrics['mape'])
transformer_r2 = float(transformer_metrics['r2_score'])

print('\nTransformer Model Performance:')
print(f'MAE:   {transformer_mae:.4f}')
print(f'RMSE:  {transformer_rmse:.4f}')
print(f'MAPE:  {transformer_mape:.4f}%')
print(f'R² Score: {transformer_r2:.4f}')

plot_history(transformer_history, 'Transformer')
plot_predictions(transformer_metrics['y_true'], transformer_metrics['y_pred'], 'Transformer: Actual vs Predicted Temperature')
plot_residuals(transformer_metrics['residuals'], 'Transformer')

In [ ]:
comparison_df = pd.DataFrame({
    'Metric': ['MAE', 'RMSE', 'MAPE (%)', 'R² Score', 'Training Time (s)', 'Parameters'],
    'RNN (LSTM/GRU)': [
        rnn_mae,
        rnn_rmse,
        rnn_mape,
        rnn_r2,
        rnn_training_time,
        rnn_total_parameters,
    ],
    'Transformer': [
        transformer_mae,
        transformer_rmse,
        transformer_mape,
        transformer_r2,
        transformer_training_time,
        transformer_total_parameters,
    ],
})

print(comparison_df.to_string(index=False))

metric_subset = comparison_df.iloc[:4].copy()
metric_long = metric_subset.melt(id_vars='Metric', var_name='Model', value_name='Value')
plt.figure(figsize=(9, 5))
sns.barplot(data=metric_long, x='Metric', y='Value', hue='Model', palette='Set2')
plt.title('Forecasting Metrics Comparison')
plt.show()

plt.figure(figsize=(14, 5))
limit = min(220, len(rnn_metrics['y_true']))
plt.plot(rnn_metrics['y_true'][:limit], label='Actual', linewidth=2)
plt.plot(rnn_metrics['y_pred'][:limit], label='RNN Prediction', linewidth=2)
plt.plot(transformer_metrics['y_pred'][:limit], label='Transformer Prediction', linewidth=2)
plt.title('Prediction Comparison on Test Window')
plt.xlabel('Test Time Step')
plt.ylabel('Temperature')
plt.legend()
plt.show()

plt.figure(figsize=(10, 4))
epochs_rnn = range(1, len(rnn_history['train_loss']) + 1)
epochs_transformer = range(1, len(transformer_history['train_loss']) + 1)
plt.plot(epochs_rnn, rnn_history['val_loss'], marker='o', label='RNN Validation Loss')
plt.plot(epochs_transformer, transformer_history['val_loss'], marker='o', label='Transformer Validation Loss')
plt.title('Validation Loss Comparison')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.legend()
plt.show()

def generate_analysis_text():
    better_model = 'RNN' if rnn_rmse <= transformer_rmse else 'Transformer'
    rmse_gap = abs(rnn_rmse - transformer_rmse)
    faster_model = 'RNN' if rnn_training_time <= transformer_training_time else 'Transformer'
    return (
        f"{better_model} achieved the lower RMSE, beating the other model by {rmse_gap:.3f} temperature units on the test set. "
        f"The stacked LSTM processes sequences recurrently, which helps it build local temporal context from step to step, while the Transformer evaluates all positions through self-attention and can model wider context in parallel. "
        f"Attention gives the Transformer a direct path to long-range relationships, avoiding the same vanishing-gradient pressure that recurrent models can face over long histories. "
        f"The LSTM was lighter with {rnn_total_parameters:,} parameters, whereas the Transformer used {transformer_total_parameters:,}, so the Transformer paid a higher complexity cost for its flexibility. "
        f"In this run, {faster_model} trained faster. The loss curves show whether the Transformer’s extra capacity improved convergence enough to justify that additional compute."
    )

analysis_text = generate_analysis_text()
analysis_word_count = len(analysis_text.split())

print('ANALYSIS')
print(analysis_text)
print(f'Analysis word count: {analysis_word_count} words')
if analysis_word_count > 200:
    print('Warning: Analysis exceeds 200 words (guideline)')
else:
    print('Analysis within word count guideline')

In [ ]:
def get_assignment_results():
    framework_used = 'pytorch'

    results = {
        'dataset_name': dataset_name,
        'dataset_source': dataset_source,
        'n_samples': n_samples,
        'n_features': n_features,
        'sequence_length': sequence_length,
        'prediction_horizon': prediction_horizon,
        'problem_type': problem_type,
        'primary_metric': primary_metric,
        'metric_justification': metric_justification,
        'train_samples': train_samples,
        'test_samples': test_samples,
        'train_test_ratio': train_test_ratio,
        'rnn_model': {
            'framework': framework_used,
            'model_type': rnn_model_type,
            'architecture': {
                'n_layers': rnn_n_layers,
                'hidden_units': rnn_hidden_units,
                'total_parameters': rnn_total_parameters,
            },
            'training_config': {
                'learning_rate': rnn_learning_rate,
                'n_epochs': rnn_epochs,
                'batch_size': rnn_batch_size,
                'optimizer': rnn_optimizer_name,
                'loss_function': rnn_loss_name,
            },
            'initial_loss': rnn_initial_loss,
            'final_loss': rnn_final_loss,
            'training_time_seconds': rnn_training_time,
            'mae': rnn_mae,
            'rmse': rnn_rmse,
            'mape': rnn_mape,
            'r2_score': rnn_r2,
        },
        'transformer_model': {
            'framework': framework_used,
            'architecture': {
                'n_layers': transformer_n_layers,
                'n_heads': transformer_n_heads,
                'd_model': transformer_d_model,
                'd_ff': transformer_d_ff,
                'has_positional_encoding': True,
                'has_attention': True,
                'total_parameters': transformer_total_parameters,
            },
            'training_config': {
                'learning_rate': transformer_learning_rate,
                'n_epochs': transformer_epochs,
                'batch_size': transformer_batch_size,
                'optimizer': transformer_optimizer_name,
                'loss_function': transformer_loss_name,
            },
            'initial_loss': transformer_initial_loss,
            'final_loss': transformer_final_loss,
            'training_time_seconds': transformer_training_time,
            'mae': transformer_mae,
            'rmse': transformer_rmse,
            'mape': transformer_mape,
            'r2_score': transformer_r2,
        },
        'analysis': analysis_text,
        'analysis_word_count': analysis_word_count,
        'rnn_loss_decreased': rnn_final_loss < rnn_initial_loss,
        'transformer_loss_decreased': transformer_final_loss < transformer_initial_loss,
    }

    return results

assignment_results = get_assignment_results()
print('ASSIGNMENT RESULTS SUMMARY')
print(json.dumps(assignment_results, indent=2))

In [ ]:
print('ENVIRONMENT INFORMATION')
print(f'Python version: {platform.python_version()}')
print(f'PyTorch version: {torch.__version__}')
print(f'Platform: {platform.platform()}')
print('\nREQUIRED: Add a screenshot of your Google Colab or BITS Virtual Lab account details below this cell before submission.')

## Final Checklist
- Replace the student information placeholders.
- Rename the file to match your real BITS ID.
- Run Kernel -> Restart & Run All so every output is visible.
- Add your environment screenshot in a markdown cell after the environment section.
- Verify the JSON summary prints without errors before submission.